In [1]:

import os, time, copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torchvision.models import densenet121
from sklearn.model_selection import train_test_split

from torch.ao.quantization import QConfig, QConfigMapping
from torch.ao.quantization.observer import MinMaxObserver, PerChannelMinMaxObserver
from torch.ao.quantization.quantize_fx import prepare_fx, convert_fx

torch.backends.quantized.engine = "fbgemm"
torch.manual_seed(42)
np.random.seed(42)

#  Data
transform = transforms.Compose([
    transforms.Resize((224, 224), antialias=True),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])
data_dir = "DIR/horse/sapi279d-test_workspace/train"
dataset = datasets.ImageFolder(root=data_dir, transform=transform)
NUM_CLASSES = len(dataset.classes)

targets = np.array(dataset.targets)
train_idx, val_idx, test_idx = [], [], []
for c in np.unique(targets):
    idxs = np.where(targets == c)[0]
    tr, tmp = train_test_split(idxs, test_size=0.15, random_state=42, shuffle=True)
    va, te = train_test_split(tmp, test_size=1/3, random_state=42, shuffle=True)
    train_idx.extend(tr); val_idx.extend(va); test_idx.extend(te)

train_dataset = Subset(dataset, train_idx)
val_dataset   = Subset(dataset, val_idx)
test_dataset  = Subset(dataset, test_idx)

batch_size = 128
num_workers = min(8, os.cpu_count() or 1)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,  num_workers=num_workers, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
print(f"Total images: {len(dataset)}, Classes: {NUM_CLASSES}")
print(" DataLoaders ready")

# Load FP32 (same head)
ckpt_path = "densenet121_weights.pth"
state = torch.load(ckpt_path, map_location="cpu")
state = {k.replace("_orig_mod.", ""): v for k, v in state.items()}

model_fp32 = densenet121(weights=None)
model_fp32.classifier = nn.Linear(model_fp32.classifier.in_features, NUM_CLASSES)
missing, unexp = model_fp32.load_state_dict(state, strict=False)
print("Missing keys:", list(missing))
print("Unexpected keys:", list(unexp))
model_fp32.eval()
print(" FP32 DenseNet121 loaded")

# Per-channel Symmetric qconfig (FX)
per_channel_qconfig = QConfig(
    activation=MinMaxObserver.with_args(dtype=torch.quint8, qscheme=torch.per_tensor_affine),
    weight=PerChannelMinMaxObserver.with_args(dtype=torch.qint8, qscheme=torch.per_channel_symmetric),
)
qconfig_mapping = QConfigMapping().set_global(per_channel_qconfig)

example_inputs = (torch.randn(1, 3, 224, 224),)
prepared = prepare_fx(model_fp32, qconfig_mapping, example_inputs)

def calibrate(model, loader, num_batches=20):
    model.eval()
    with torch.inference_mode():
        for i, (x, _) in enumerate(loader):
            if i >= num_batches: break
            model(x)

calibrate(prepared, val_loader, num_batches=20)
print("Calibration done ")

model_int8 = convert_fx(prepared).to("cpu").eval()
print("Per-channel symmetric INT8 ready ")

#  Eval, size, runtime
def evaluate(model, loader):
    model.eval()
    tot, corr = 0, 0
    with torch.inference_mode():
        for x, y in loader:
            out = model(x)
            pred = out.argmax(1)
            corr += (pred == y).sum().item()
            tot += y.size(0)
    return 100.0 * corr / tot

def get_model_size_mb(model, fname="__tmp__.pth"):
    torch.save(model.state_dict(), fname)
    mb = os.path.getsize(fname) / 1e6
    os.remove(fname)
    return mb

def benchmark(model, loader, num_batches=50):
    model.eval()
    t0 = time.time()
    with torch.inference_mode():
        for i, (x, _) in enumerate(loader):
            if i >= num_batches: break
            _ = model(x)
    t1 = time.time()
    imgs = num_batches * loader.batch_size
    return (t1 - t0)/num_batches*1000.0, (t1 - t0)/imgs*1000.0

acc_fp32 = evaluate(model_fp32, test_loader)
acc_int8 = evaluate(model_int8,  test_loader)
print(f"FP32 Accuracy: {acc_fp32:.2f}%")
print(f"INT8 Accuracy (Per-channel): {acc_int8:.2f}%")

sz_fp32 = get_model_size_mb(model_fp32)
sz_int8 = get_model_size_mb(model_int8)
print(f"FP32 size: {sz_fp32:.2f} MB | INT8 size: {sz_int8:.2f} MB")

b_fp32, i_fp32 = benchmark(model_fp32, test_loader)
b_int8, i_int8 = benchmark(model_int8,  test_loader)
print(f"FP32 Inference: {b_fp32:.2f} ms/batch | {i_fp32:.2f} ms/image")
print(f"INT8 Inference: {b_int8:.2f} ms/batch | {i_int8:.2f} ms/image")



/software/util/JupyterLab/alpha/share/pytorch_v2/lib/python3.10/site-packages/torch/utils/data/dataloader.py:558: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


Total images: 100000, Classes: 200
 DataLoaders ready
Missing keys: []
Unexpected keys: []
 FP32 DenseNet121 loaded


/software/util/JupyterLab/alpha/share/pytorch_v2/lib/python3.10/site-packages/torch/utils/data/dataloader.py:558: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


Calibration done 
Per-channel symmetric INT8 ready 
FP32 Accuracy: 47.84%
INT8 Accuracy (Per-channel): 29.20%
FP32 size: 29.23 MB | INT8 size: 8.18 MB
FP32 Inference: 10549.25 ms/batch | 82.42 ms/image
INT8 Inference: 7471.27 ms/batch | 58.37 ms/image
